### 88. Merge Sorted Array
You are given two integer arrays `nums1` and `nums2`, sorted in **non-decreasing order**, and two integers `m` and `n`, representing the number of elements in `nums1` and `nums2` respectively.

**Merge** `nums1` and `nums2` into a single array sorted in **non-decreasing order**.

The final sorted array should not be returned by the function, but instead be _stored inside the array_ `nums1`. To accommodate this, `nums1` has a length of `m + n`, where the first `m` elements denote the elements that should be merged, and the last `n` elements are set to `0` and should be ignored. `nums2` has a length of `n`.

Example 1:
```shell
Input: nums1 = [1,2,3,0,0,0], m = 3, nums2 = [2,5,6], n = 3
Output: [1,2,2,3,5,6]
Explanation: The arrays we are merging are [1,2,3] and [2,5,6].
The result of the merge is [1,2,2,3,5,6] with the underlined elements coming from nums1.
```
Example 2:
```shell
Input: nums1 = [1], m = 1, nums2 = [], n = 0
Output: [1]
Explanation: The arrays we are merging are [1] and [].
The result of the merge is [1].
```
Example 3:
```shell
Input: nums1 = [0], m = 0, nums2 = [1], n = 1
Output: [1]
Explanation: The arrays we are merging are [] and [1].
The result of the merge is [1].
Note that because m = 0, there are no elements in nums1. The 0 is only there to ensure the merge result can fit in nums1.
```

**Constraints**:
```shell
nums1.length == m + n
nums2.length == n
0 <= m, n <= 200
1 <= m + n <= 200
-10^9 <= nums1[i], nums2[j] <= 10^9
```

Follow up: Can you come up with an algorithm that runs in O(m + n) time?

### Reasoning
#### Step 1 - Gather Requirements
Given 2 integer arrays **sorted** in ASC order `nums1` and `nums2`, integer `m` representing the number of elements in `nums1` and `n` representing the number of elements in `nums2`. Also, `nums1` contains `n` zeros after its non-zero items that must be **ignored**, the zeros only needed to placehold for `nums2` length. I need to merge the 2 arrays into `nums1` and maintain the sorted order.

- **Input**: 2 integer arrays, 2 integers
- **Output**: a sorted array of integers
- `nums1` can not be fully empty, in this case it contains `0`
- `nums2` can be fully empty
- `m`, `n` could be `0` up to `200`
- elements of `nums1` and `nums2` are from `-10^9` to `10^9`
- Arrays are sorted in ASC
- Duplicates & negatives are possible
- The order of the output matters and must be ASC

#### Step 2 - Edge cases + Examples
- **Case 1**: `nums1 = [0], nums2 = [], m = 1, n = 0` → output `[0]`. (The `0` is real data since `n=0` means no placeholder slots.)
- **Case 2** (already correct): `nums1 = [0], nums2 = [0], m = 0, n = 1` → output `[0]`.
- **Case 3** (already correct): the general merge example → `[1,2,2,3,3,4,5]`.

> `m=0, n=0` and `nums1=[], nums2=[0], m=0, n=0`: both illegal. `1 <= m + n <= 200` forbids `m` and `n` being `0` simultaneously.

Your valid edge-case set is: `m=0` (`nums1` fully placeholder), `n=0` (`nums2` fully empty), and the general interleaved case.
1. `nums1 = [0], nums2 = [], m = 1, n = 0 -> [0]` (`0` is real data)
2. `nums1 = [0], nums2 = [0], m = 0, n = 1 -> [0]`
3. `nums1 = [1, 2, 3, 3, 0, 0, 0], nums2 = [2, 4, 5], m = 4, n = 3 -> [1, 2, 2, 3, 3, 4, 5]`

#### Step 3 - Brute Force
**Extra list for result + copy non-zero items + sort them**.

Ignoring the **in-place** requirement & using the **extra containter for the result**, store `m` & `n` items from both arrays, sort them and return.
- **Time**: $O(m)$ for storing `nums1` items + $O(n)$ for storing `nums2` items. Sort the resulting array `result` takes $O(n\space log\space n)$, and dominates.
- **Space**: $O(m\space +\space n)$ to accomodate copies from both arrays.

[Visual](../images/88merge_sorted_array_brute.png)

In [ ]:
def func(nums1: list[int], m: int, nums2: list[int], n: int) -> list[int]:
    # brute force - extra list + sort + slice assignment to original nums1
    # setup
    # set temp container to hold elements being merged: temp = []

    # 1. scan nums1 - for loop - full pass - add 'm' items to temp list
    # 2. scan nums2 - for loop - full pass - add 'n' items to temp list
    # 3. sort temp list
    # 4. store temp elements in original nums1 container (use slice assignment)
    # 5. return nums1

    temp = []  # space O(m + n)

    temp.extend(nums1[:m])  # O(m)
    temp.extend(nums2[:n])  # O(n)

    nums1[:] = sorted(temp)  # O(n log n)

    return nums1

In [ ]:
# func([1,2,3,0,0,0], 3, [2,5,6], 3)  # [1,2,2,3,5,6]
# func([1], 1, [], 0)  # [1]
func([0], 0, [1], 1)  # [1]

#### Step 4 - Select Algorithm
Two separate arrays being merged into one.

Two problems to solve from Brute force solution:
1. Get rid of the **extra container** `temp` (extra space $O(m\space +\space n)$)
2. Get rid of **sorting** - unnecessary, the arrays are already sorted

Three pointers — `x`, `y`, and a **write** pointer `i`. Which direction do all three move, and why start from the back?
- All pointers are moving **backwards (to left) starting from the ends** of arrays since we use `0` placeholders to insert the **greatest** item from 2 numeric arrays when compare them.
- So, we **insert the biggest** item and move writing pointer backwards as well as the other 2 pointer that compare items.
- We have 2 cases to stop:
  1. when the first (`n1`) array is done - it means the remaining items from second (`n2`) array are smaller and must be placed **in front of** the first array, in order, moving from the back to the beginning of the array with writing pointer `i`;
  2. when the second array is done - we just leave everything as is, since the first array is already sorted.

Both termination cases are right.

> The key insight you named — "start from the back because the placeholders are already there" — is exactly what makes this $O(1)$ space instead of $O(n)$. Writing forward would **overwrite elements** you haven't compared yet.

#### Stop Logic
The stop conditions come from asking one question after you have the basic comparison working:
> **"what happens when one array runs out before the other?"**

There are two cases:
```shell
CASE A — y < 0 (n2 is exhausted):
  n2's elements are all written. n1's remaining elements
  are already sitting in their correct positions — you put
  them there originally. Nothing to do. Break.

CASE B — x < 0 (n1's valid elements are exhausted):
  n1's elements are all written. n2's remaining elements
  must fill the remaining slots. Keep writing from y.
```
**Why they go BEFORE the comparison, not after:**

If `x = -1` and you reach the comparison `n1[x] > n2[y]`, Python doesn't crash — it silently reads `n1[-1]`, which is the _last element of n1_. Wrong value, no error. You'd write garbage without knowing it.
> The guard must come first to prevent the comparison from ever running on an invalid index.

`continue` — it IS needed there. Without it, after writing `n2[y]` when `x < 0`, the code falls through to `if n1[x] > n2[y]` with `x` still at `-1`. The guard only skips the fall-through if you explicitly `continue`. Remove it and you get the exact bug the guard was meant to prevent.

[Visual](../images/88merge_sorted_array.png)

[Visual Stop When nums1 is done](../images/88merge_sorted_array_stop1.png)

[Visual Stop When nums2 is done](../images/88merge_sorted_array_stop2.png)

#### Step 5, 6 - Signature + Pseudocode (Skeleton)
```py
def func(nums1: list[int], m: int, nums2: list[int], n: int) -> list[int]:
    # three pointers - one pass - multipointers across 2 arrays + scan from end
    # setup
    # set 3 pointers:
    #   1) at the end of nums1 elements: x = m - 1
    #   2) at the end of nums2 elements;  y = n - 1
    #   3) at the end of placeholders in nums1: i - for loop - full pass reversly (write)

    # 1. scan both arrays with 3 pointers, compare x vs y:
    #   x > y: i = x
    #   y >= x: i = y
    # 2. stop logic:
    #   x < 0: nums1 items done, move all nums2 items in front of nums1: i = y
    #   y < 0: nums2 items done, leave as is since nums1 items are in place already: break
    # 3. return nums1
```

#### Step 7 - Write Code

In [3]:
def merge(nums1, m, nums2, n):
    x, y = m - 1, n - 1

    for i in range(len(nums1) - 1, -1, -1):
        # stop logic - pointer out of bounds
        if x < 0:  # nums1 done — drain remaining nums2
            nums1[i] = nums2[y]
            y -= 1
            continue  # skips the fall-through
        if y < 0:  # nums2 done — nums1 elements already in place
            break

        # comparison logic - happy path
        if nums1[x] > nums2[y]:
            nums1[i] = nums1[x]
            x -= 1
        else:
            nums1[i] = nums2[y]
            y -= 1

    return nums1

#### Step 8 - Trace through + Edge cases

In [ ]:
# merge([4, 7, 8, 9, 0, 0, 0], 4, [2, 5, 6], 3)
# merge([2, 5, 9, 30, 0, 0, 0], 4, [45, 49, 59], 3)
# merge([100, 200, 300], 3, [1, 2, 3], 3)
# merge([1, 2, 3], 3, [30, 40, 50], 3)

# func([1,2,3,0,0,0], 3, [2,5,6], 3)  # [1,2,2,3,5,6]
# func([1], 1, [], 0)  # [1]
func([0], 0, [1], 1)  # [1]

[1]

#### Step 9 - Complexity Analisys
- **Time**: $O(m + n)$ one pass. We are traversing 2 input arrays once from the end.
- **Space**: $O(1)$ no extra space is used.

#### The mental model to file

> Stop conditions exist because pointers can go out of bounds. You add them when you ask "what breaks?" after the happy path works. They always go before any operation that uses those pointers.

**Two Pointer pattern summary — you're solid on:**

|Flavor|Problem|Direction|Stop when|
|---|---|---|---|
|Opposite ends|Two Sum II, Valid Palindrome|converge|`left < right`|
|Same direction|Move Zeros, Remove Duplicates|both right|`fast == len`|
|Multi-pointer|Merge Sorted Array|all backward|each sub-array exhausted|

---

### Automated Tests — pytest/ipytest blueprint

Manually printing a result and eyeballing it (what Step 8 did above) doesn't scale once you've written 20 solutions — you either retest everything by hand every session, or you trust old code without rechecking it. This section replaces manual printing with real `pytest` tests, run inline via `ipytest`, so every notebook from here on gets the same three-category structure Step 8 already asks for:

- **General case** — the normal input
- **Extreme cases** — the edge cases from Step 2 (`m=0`, `n=0`, duplicates across both arrays)
- **User error** — malformed input; what *should* happen?

Run the setup cell once per notebook (anywhere before the test cell), then the test cell itself must start with the `%%ipytest` magic as its first line.

In [1]:
import ipytest

ipytest.autoconfig()

In [9]:
%%ipytest

import pytest


@pytest.mark.parametrize("nums1, m, nums2, n, expected", [
    pytest.param([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3, [1, 2, 2, 3, 5, 6], id="general_case"),
    pytest.param([1], 1, [], 0, [1], id="extreme_nums2_empty"),
    pytest.param([0], 0, [1], 1, [1], id="extreme_nums1_all_placeholder"),
    pytest.param([2, 2, 3, 0, 0], 3, [2, 5], 2, [2, 2, 2, 3, 5], id="extreme_duplicates_across_both"),
])
def test_merge_general_and_extreme(nums1, m, nums2, n, expected):
    assert merge(nums1, m, nums2, n) == expected

....                                                                                         [100%]
4 passed in 3.00s


**User error category — intentionally skipped here.** LeetCode guarantees `nums1.length == m + n` and `nums2.length == n` for every input to this problem; a call that violates that (e.g. `m`/`n` not matching the actual array lengths) is outside the problem's contract, not a case the solution is expected to defend against. No test added for it — this category only gets a test on problems where malformed input is actually possible under the stated constraints (e.g. the caller passing a negative `k`, or a type the signature doesn't guard against).